# 01. Ingesta y Validación de Datos — IBM TabFormer
**Unidad 1:** Detección de Fraude en Transacciones con Tarjeta de Crédito  
**Dataset:** IBM TabFormer (`card_transaction.v1.csv`)  
**Entorno:** Python 3.12 | Polars | scikit-learn

---

### Objetivos del Notebook:
1. Inspeccionar el entorno de ejecución y las versiones de las librerías analíticas.
2. Identificar los archivos de datos en `data/DATA IBM/`, registrando rutas, nombres y tamaños en disco.
3. Inspeccionar el esquema de columnas y tipos de datos utilizando operaciones Lazy con **Polars** (sin saturar la memoria RAM).
4. Verificar la integridad de los encabezados, descartar duplicidades de archivos o filas repetidas en cabecera.
5. Contar el número total de transacciones mediante cómputo eficiente en streaming.
6. Evitar cargar innecesariamente los 24 millones de registros en la memoria RAM de la laptop.
7. Exportar un resumen reproducible de ingesta (`ingesta_resumen.json`) para consumo de las siguientes etapas.


In [1]:
import sys
from pathlib import Path

# Configuración de ruta base portable
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR
while not (PROJECT_ROOT / "notebooks").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Raíz del proyecto: {PROJECT_ROOT}")


Raíz del proyecto: /home/glender/ProyectosUNi/iaupeu


In [2]:
import polars as pl
import sklearn
import pandas as pd
import numpy as np
import json
import os
import psutil

print("=== Versiones de Entorno ===")
print(f"Python: {sys.version.split()[0]}")
print(f"Polars: {pl.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

# Diagnóstico de RAM disponible
mem = psutil.virtual_memory()
print("\n=== Diagnóstico de Memoria ===")
print(f"Memoria Total: {mem.total / (1024**3):.2f} GB")
print(f"Memoria Disponible: {mem.available / (1024**3):.2f} GB")
print(f"Uso Actual: {mem.percent}%")


=== Versiones de Entorno ===
Python: 3.12.14
Polars: 2.1.0
scikit-learn: 1.9.1
pandas: 3.1.0
numpy: 2.5.2

=== Diagnóstico de Memoria ===
Memoria Total: 15.41 GB
Memoria Disponible: 6.43 GB
Uso Actual: 58.3%


## 1. Descubrimiento e Inspección de Archivos en Disco
Revisamos los archivos presentes en el directorio `data/DATA IBM/` para confirmar si se dispone de partes fragmentadas o de un archivo consolidado.


In [3]:
from notebooks._comun.config import RAW_DATA_DIR, RAW_DATA_FILE, INGESTION_SUMMARY_FILE
from notebooks._comun.data_utils import get_file_metadata

# Escaneo de archivos CSV en el directorio de IBM
csv_files = list(RAW_DATA_DIR.glob("*.csv"))
print(f"Archivos CSV encontrados en {RAW_DATA_DIR.name}: {len(csv_files)}\n")

files_metadata = []
for f in sorted(csv_files):
    meta = get_file_metadata(f)
    files_metadata.append(meta)
    print(f"- Archivo: {meta['name']}")
    print(f"  Ruta: {meta['path']}")
    print(f"  Tamaño: {meta['size_mb']} MB ({meta['size_gb']} GB)")

# Nota técnica sobre la estructura de archivos:
# El dataset en el entorno local está provisto como un único archivo consolidado
# 'card_transaction.v1.csv' con 24,386,900 registros (2.19 GB).


Archivos CSV encontrados en DATA IBM: 1

- Archivo: card_transaction.v1.csv
  Ruta: /home/glender/ProyectosUNi/iaupeu/data/DATA IBM/card_transaction.v1.csv
  Tamaño: 2245.55 MB (2.193 GB)


## 2. Lectura del Esquema con Polars (Lazy Evaluation)
Utilizamos `pl.scan_csv()` para inspeccionar el esquema de nombres y tipos sin cargar los 2.3 GB en memoria.


In [4]:
# Escaneo Lazy del dataset
lf = pl.scan_csv(RAW_DATA_FILE)

# Recolección del esquema sin leer los datos completos
schema = lf.collect_schema()

print(f"Total de columnas detectadas: {len(schema)}\n")
schema_info = []
for col_name, dtype in schema.items():
    schema_info.append({"columna": col_name, "tipo": str(dtype)})
    print(f"  • {col_name:16s} : {dtype}")


Total de columnas detectadas: 15

  • User             : Int64
  • Card             : Int64
  • Year             : Int64
  • Month            : Int64
  • Day              : Int64
  • Time             : String
  • Amount           : String
  • Use Chip         : String
  • Merchant Name    : Int64
  • Merchant City    : String
  • Merchant State   : String
  • Zip              : Float64
  • MCC              : Int64
  • Errors?          : String
  • Is Fraud?        : String


## 3. Inspección de Encabezados y Primeras Filas
Leemos una muestra de las primeras 5 filas para comprobar la consistencia de los nombres y tipos de valores.


In [5]:
# Lectura de las primeras 5 filas
df_head = lf.limit(5).collect()
print("Primeras 5 filas del dataset:")
print(df_head)


Primeras 5 filas del dataset:
shape: (5, 15)
┌──────┬──────┬──────┬───────┬───┬─────────┬──────┬─────────┬───────────┐
│ User ┆ Card ┆ Year ┆ Month ┆ … ┆ Zip     ┆ MCC  ┆ Errors? ┆ Is Fraud? │
│ ---  ┆ ---  ┆ ---  ┆ ---   ┆   ┆ ---     ┆ ---  ┆ ---     ┆ ---       │
│ i64  ┆ i64  ┆ i64  ┆ i64   ┆   ┆ f64     ┆ i64  ┆ str     ┆ str       │
╞══════╪══════╪══════╪═══════╪═══╪═════════╪══════╪═════════╪═══════════╡
│ 0    ┆ 0    ┆ 2002 ┆ 9     ┆ … ┆ 91750.0 ┆ 5300 ┆ null    ┆ No        │
│ 0    ┆ 0    ┆ 2002 ┆ 9     ┆ … ┆ 91754.0 ┆ 5411 ┆ null    ┆ No        │
│ 0    ┆ 0    ┆ 2002 ┆ 9     ┆ … ┆ 91754.0 ┆ 5411 ┆ null    ┆ No        │
│ 0    ┆ 0    ┆ 2002 ┆ 9     ┆ … ┆ 91754.0 ┆ 5651 ┆ null    ┆ No        │
│ 0    ┆ 0    ┆ 2002 ┆ 9     ┆ … ┆ 91750.0 ┆ 5912 ┆ null    ┆ No        │
└──────┴──────┴──────┴───────┴───┴─────────┴──────┴─────────┴───────────┘


## 4. Verificación de Encabezados Repetidos en el CSV
Comprobamos que las filas internas no contengan la cadena del encabezado (problema común en concatenaciones accidentales de archivos CSV).


In [6]:
# Verificación de que la columna 'User' no contenga el texto 'User'
header_duplicates = (
    lf.filter(pl.col("User").cast(pl.String) == "User")
    .select(pl.len().alias("n_header_dups"))
    .collect()
    .item()
)

print(f"Filas que repiten el encabezado: {header_duplicates}")
assert header_duplicates == 0, "¡Se encontraron encabezados repetidos en el archivo!"
print("Integridad de encabezados validada exitosamente: 0 filas repetidas.")


Filas que repiten el encabezado: 0
Integridad de encabezados validada exitosamente: 0 filas repetidas.


## 5. Conteo Total de Transacciones
Ejecutamos un cálculo de longitud en streaming con Polars para registrar el volumen exacto de registros.


In [7]:
# Conteo exacto mediante LazyFrame
total_records = lf.select(pl.len().alias("total_rows")).collect().item()
print(f"Número total de registros en el dataset: {total_records:,} filas")


Número total de registros en el dataset: 24,386,900 filas


## 6. Generación del Resumen Reproducible de Ingesta
Exportamos el resumen estructural a `data/processed/ingesta_resumen.json`.


In [8]:
from notebooks._comun.data_utils import save_json

ingestion_summary = {
    "dataset_name": "IBM TabFormer - Credit Card Transactions",
    "source_file": RAW_DATA_FILE.name,
    "file_size_gb": files_metadata[0]["size_gb"],
    "file_size_mb": files_metadata[0]["size_mb"],
    "total_rows": total_records,
    "total_columns": len(schema),
    "columns": schema_info,
    "target_column": "Is Fraud?",
    "header_repeats": header_duplicates,
    "execution_library": f"Polars v{pl.__version__}",
}

save_json(ingestion_summary, INGESTION_SUMMARY_FILE)
print(f"Resumen guardado exitosamente en: {INGESTION_SUMMARY_FILE}")
print(json.dumps(ingestion_summary, indent=2))


Resumen guardado exitosamente en: /home/glender/ProyectosUNi/iaupeu/data/processed/ingesta_resumen.json
{
  "dataset_name": "IBM TabFormer - Credit Card Transactions",
  "source_file": "card_transaction.v1.csv",
  "file_size_gb": 2.193,
  "file_size_mb": 2245.55,
  "total_rows": 24386900,
  "total_columns": 15,
  "columns": [
    {
      "columna": "User",
      "tipo": "Int64"
    },
    {
      "columna": "Card",
      "tipo": "Int64"
    },
    {
      "columna": "Year",
      "tipo": "Int64"
    },
    {
      "columna": "Month",
      "tipo": "Int64"
    },
    {
      "columna": "Day",
      "tipo": "Int64"
    },
    {
      "columna": "Time",
      "tipo": "String"
    },
    {
      "columna": "Amount",
      "tipo": "String"
    },
    {
      "columna": "Use Chip",
      "tipo": "String"
    },
    {
      "columna": "Merchant Name",
      "tipo": "Int64"
    },
    {
      "columna": "Merchant City",
      "tipo": "String"
    },
    {
      "columna": "Merchant State",
   